In [1]:
import tensorflow as tf
import numpy as np

from tensorflow import keras
from tensorflow.keras import Sequential, layers

In [2]:
train_dir = "/kaggle/input/datasets/ayush1220/cifar10/cifar10/train"
test_dir  = "/kaggle/input/datasets/ayush1220/cifar10/cifar10/test"

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

Found 50000 files belonging to 10 classes.


I0000 00:00:1789913418.014036      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789913418.017127      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 10000 files belonging to 10 classes.


In [3]:
class_names = train_ds.class_names
print(class_names)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
x_train_list = []
y_train_list = []

x_test_list = []
y_test_list = []


for imgs, labels in train_ds:
    x_train_list.append(imgs.numpy())             # converting imgs into numpy array
    y_train_list.append(labels.numpy())

for imgs, labels in test_ds:
    x_test_list.append(imgs.numpy())
    y_test_list.append(labels.numpy())

In [5]:
print(len(x_train_list))
print(len(x_test_list))

782
157


In [6]:
x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

x_test = np.concatenate(x_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# -----------------------------------------------------------------------------

print(x_train.shape)
print(x_test.shape)


(50000, 32, 32, 3)
(10000, 32, 32, 3)


In [7]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# -----------------------------------------------------------------------------

print(x_train.shape)
print(x_test.shape)

(50000, 32, 32, 3)
(10000, 32, 32, 3)


### Sequential

In [ ]:
# STEP 1
model = keras.Sequential([
    keras.Input(shape=(32,32,3)),
    layers.Conv2D(32, 3, padding='valid', activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Conv2D(64,3,activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,3,activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10)
])


# STEP 2
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(3e-4),
    metrics=['accuracy']
)


# STEP 3
model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=10,
    verbose=2
)


# STEP 4
model.evaluate(
    x_test, y_test,
    batch_size=64,
    verbose=2
)

Epoch 1/10


I0000 00:00:1789913517.021113     135 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


782/782 - 9s - 11ms/step - accuracy: 0.3832 - loss: 1.6900
Epoch 2/10
782/782 - 3s - 3ms/step - accuracy: 0.5201 - loss: 1.3502
Epoch 3/10
782/782 - 3s - 3ms/step - accuracy: 0.5711 - loss: 1.2215
Epoch 4/10
782/782 - 3s - 3ms/step - accuracy: 0.6016 - loss: 1.1311
Epoch 5/10
782/782 - 3s - 3ms/step - accuracy: 0.6329 - loss: 1.0571
Epoch 6/10
782/782 - 3s - 3ms/step - accuracy: 0.6536 - loss: 0.9944
Epoch 7/10
782/782 - 3s - 3ms/step - accuracy: 0.6722 - loss: 0.9421
Epoch 8/10
782/782 - 3s - 4ms/step - accuracy: 0.6907 - loss: 0.8930
Epoch 9/10
782/782 - 3s - 3ms/step - accuracy: 0.7086 - loss: 0.8444
Epoch 10/10
782/782 - 3s - 3ms/step - accuracy: 0.7192 - loss: 0.8114
157/157 - 1s - 9ms/step - accuracy: 0.6867 - loss: 0.9077


[0.9076914191246033, 0.6866999864578247]

### Functional

In [9]:
def my_model():
    inputs = keras.Input(shape=(32,32,3))
    x = layers.Conv2D(32,3)(inputs)
    x = layers.BatchNormalization()(x)
    x = keras.activations.relu(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 5, padding='same')(x)
    x = keras.activations.relu(x)
    x = layers.Conv2D(128, 3)(x)
    x = layers.BatchNormalization()(x)
    x = keras.activations.relu(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(10)(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    return model

# STEP 1
mdoel = my_model()


# STEP 2
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(3e-4),
    metrics=['accuracy']
)


# STEP 3
model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=10,
    verbose=2
)


# STEP 4
model.evaluate(
    x_test, y_test,
    batch_size=64,
    verbose=2
)
    

Epoch 1/10
782/782 - 6s - 8ms/step - accuracy: 0.7320 - loss: 0.7759
Epoch 2/10
782/782 - 3s - 3ms/step - accuracy: 0.7433 - loss: 0.7422
Epoch 3/10
782/782 - 3s - 3ms/step - accuracy: 0.7563 - loss: 0.7132
Epoch 4/10
782/782 - 3s - 3ms/step - accuracy: 0.7650 - loss: 0.6838
Epoch 5/10
782/782 - 3s - 3ms/step - accuracy: 0.7707 - loss: 0.6597
Epoch 6/10
782/782 - 3s - 4ms/step - accuracy: 0.7813 - loss: 0.6323
Epoch 7/10
782/782 - 3s - 4ms/step - accuracy: 0.7906 - loss: 0.6084
Epoch 8/10
782/782 - 3s - 4ms/step - accuracy: 0.7975 - loss: 0.5860
Epoch 9/10
782/782 - 3s - 4ms/step - accuracy: 0.8084 - loss: 0.5585
Epoch 10/10
782/782 - 3s - 4ms/step - accuracy: 0.8133 - loss: 0.5412
157/157 - 1s - 8ms/step - accuracy: 0.7312 - loss: 0.8317


[0.8316731452941895, 0.7311999797821045]

In [10]:
%load_ext cudf.pandas
import pandas as pd

y_temp = pd.Series(y_train)
y_temp.unique()

array([6, 3, 7, 1, 8, 9, 5, 2, 4, 0], dtype=int32)